# UNO-VC prediction demonstration

Choose upward or downward. The choice selects both the held-out example and the matching trained model.

In [ ]:
from pathlib import Path
import gc

import matplotlib.pyplot as plt
import numpy as np
import torch

from prediction import EXAMPLE_FILES, load_trained_model, prepare_input, reconstruct_prediction

DIRECTION = "upward"  # change to "downward" for the other model and example
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("direction:", DIRECTION)
print("device:", DEVICE)

In [ ]:
with np.load(EXAMPLE_FILES[DIRECTION], allow_pickle=False) as file:
    example = {key: file[key] for key in file.files}

data_ax = example["input_acceleration"]
data_vx = example["input_velocity"]
observed_ay = example["observed_acceleration"]
observed_vy = example["observed_velocity"]

print("record:", int(example["record_id"][0]))
print("input acceleration:", data_ax.shape)
print("input velocity:", data_vx.shape)

In [ ]:
data_x = prepare_input(data_ax, data_vx)
model = load_trained_model(DIRECTION, DEVICE)

with torch.no_grad():
    raw_output = model(data_x.to(DEVICE)).cpu().numpy()

predicted_ay, predicted_vy = reconstruct_prediction(raw_output, data_ax, data_vx)
print("raw model output:", raw_output.shape)
print("predicted acceleration:", predicted_ay.shape)
print("predicted velocity:", predicted_vy.shape)

In [ ]:
expected_ay = example["expected_acceleration"]
expected_vy = example["expected_velocity"]

max_acceleration_error = np.max(np.abs(predicted_ay - expected_ay))
max_velocity_error = np.max(np.abs(predicted_vy - expected_vy))
print("maximum acceleration error from original prediction:", max_acceleration_error)
print("maximum velocity error from original prediction:", max_velocity_error)

assert np.allclose(predicted_ay, expected_ay, atol=2e-5, rtol=2e-4)
assert np.allclose(predicted_vy, expected_vy, atol=2e-5, rtol=2e-4)
print("Prediction matches the original workflow.")

In [ ]:
time = np.linspace(0, 60, data_ax.shape[1])
input_waveforms = np.concatenate((data_ax, data_vx), axis=2)[0]
observed_waveforms = np.concatenate((observed_ay, observed_vy), axis=2)[0]
predicted_waveforms = np.concatenate((predicted_ay, predicted_vy), axis=2)[0]
channel_names = ["Acceleration Z", "Acceleration E", "Acceleration N",
                 "Velocity Z", "Velocity E", "Velocity N"]

fig, axes = plt.subplots(6, 1, figsize=(14, 14), sharex=True)
for channel, axis in enumerate(axes):
    axis.plot(time, input_waveforms[:, channel], label="input", alpha=0.65)
    axis.plot(time, observed_waveforms[:, channel], label="observed", alpha=0.8)
    axis.plot(time, predicted_waveforms[:, channel], label="predicted", alpha=0.8)
    axis.set_ylabel(channel_names[channel])
    axis.grid(alpha=0.2)
axes[0].legend(ncol=3)
axes[-1].set_xlabel("Time (s)")
fig.suptitle(DIRECTION.capitalize() + " continuation")
plt.tight_layout()
plt.show()

del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()